# 업무 테이블 규칙 시험 (test_work_rules) · 담당: 남건희
`schema_work.sql`(업무 테이블 8개)의 **한 행 단위 규칙**(PK·FK·UNIQUE·CHECK·NOT NULL)이 잘못된 입력을 막는지 확인합니다.

- **준비:** 같은 폴더에 `patent_final.db`와 `schema_work.sql`이 있어야 합니다.
- **트리거는 포함하지 않음:**

## 1. 메모리 DB 만들기 (공공 데이터 4개 + 업무 테이블 8개)

In [1]:
import sqlite3, os
SRC, SCHEMA = 'patent_final.db', 'schema_work.sql'
assert os.path.exists(SRC), 'patent_final.db가 없습니다. load_final.ipynb를 먼저 실행하세요.'
assert os.path.exists(SCHEMA), 'schema_work.sql이 같은 폴더에 없습니다.'

src = sqlite3.connect(f'file:{SRC}?mode=ro', u
ri=True)    # 원본은 읽기 전용
con = sqlite3.connect(':memory:')
src.backup(con); src.close()                               # 원본 → 메모리 복사
con.execute('PRAGMA foreign_keys = ON')                    # 외래키 검사 켜기
con.executescript(open(SCHEMA, encoding='utf-8').read())   # 업무 테이블 8개 생성

tables = [r[0] for r in con.execute("SELECT name FROM sqlite_master WHERE type='table' ORDER BY name")]
print(len(tables), '개 테이블:', ', '.join(tables))

SyntaxError: invalid syntax. Perhaps you forgot a comma? (2610243462.py, line 6)

## 2. 시험용 기본 데이터
| 표 | 넣는 내용 |
|---|---|
| team_member | 팀원 5명 |
| project | 프로젝트 1개 (담당자: 남건희) |
| criterion | 비교요소 1~4번 |
| search_term | 1번 비교요소의 검색어 'CNN', '합성곱' |
| candidate | 후보 2건 (1번: 검토 기록 있음, 2번: 검토 기록 없음) |
| candidate_reviewer | 후보 1 ← 김연준·최재원, 후보 2 ← 양재훈·박채연 |
| review | 1번: 김연준 1차 **제출**(채택, 판정 4개) / 2번: 최재원 1차 **임시저장**(판정 2개) |

In [2]:
P1, P2, P3 = [r[0] for r in con.execute('SELECT patent_id FROM patent ORDER BY patent_id LIMIT 3')]

con.executemany('INSERT INTO team_member(member_id, name, student_no, role) VALUES (?,?,?,?)', [
    (1, '김연준', '12233929', '업무 규칙·검토 기준'), (2, '최재원', '12233967', '트랜잭션·통합'),
    (3, '남건희', '12241871', '테이블·규칙·ERD'),     (4, '양재훈', '12245610', '조회 쿼리'),
    (5, '박채연', '12253388', '데이터 전처리')])
con.execute("INSERT INTO project(project_id, owner_id, title) VALUES (1, 3, '시험 프로젝트: 차량 객체 인식')")
con.executemany('INSERT INTO criterion(criterion_id, project_id, no, name) VALUES (?,1,?,?)', [
    (1, 1, '해결 문제: 객체 인식'), (2, 2, '핵심 방법: 합성곱 신경망'),
    (3, 3, '입력 데이터: 카메라 영상'), (4, 4, '적용 분야: 차량')])
con.executemany('INSERT INTO search_term(criterion_id, term) VALUES (1, ?)', [('CNN',), ('합성곱',)])
con.executemany('INSERT INTO candidate(candidate_id, project_id, patent_id, priority) VALUES (?,1,?,?)',
                [(1, P1, 3), (2, P2, 2)])
con.executemany('INSERT INTO candidate_reviewer VALUES (?,?)', [(1, 1), (1, 2), (2, 4), (2, 5)])
con.execute("INSERT INTO review VALUES (1, 1, 1, 1, 'SUBMITTED', '채택', NULL)")
con.executemany('INSERT INTO review_item VALUES (1, ?, ?, ?)', [
    (1, '○', '요약 1문장: 객체를 인식'), (2, '○', '요약 2문장: 합성곱 신경망 사용'),
    (3, '△', '요약에 영상 입력만 언급'),   (4, 'X', '차량 언급 없음')])
con.execute("INSERT INTO review(review_id, candidate_id, member_id, round) VALUES (2, 1, 2, 1)")   # 기본값 DRAFT
con.executemany('INSERT INTO review_item VALUES (2, ?, ?, ?)', [(1, '○', '요약 1문장'), (2, '△', '요약 3문장')])
con.commit()

for t in ['team_member', 'project', 'criterion', 'search_term', 'candidate', 'candidate_reviewer', 'review', 'review_item']:
    print(f'{t:20s}', con.execute(f'SELECT COUNT(*) FROM {t}').fetchone()[0])

team_member          5
project              1
criterion            4
search_term          2
candidate            2
candidate_reviewer   4
review               2
review_item          6


## 3. 잘못된 입력 23가지
| # | 잘못된 입력 | 막는 규칙 | 업무 규칙 |
|---|---|---|---|
| 1 | 학번이 8자리 숫자가 아님 | CHECK | - |
| 2 | 프로젝트 상태 '완료' | CHECK (준비·진행·확정) | 9·10 |
| 3 | 없는 팀원을 담당자로 | FK | 1 |
| 4 | 비교요소 번호 5 | CHECK (1~4) | 9 |
| 5 | 같은 프로젝트에 같은 번호 비교요소 | UNIQUE | 9 |
| 6 | 대소문자만 다른 같은 검색어('cnn') | UNIQUE + NOCASE | - |
| 7 | 빈 검색어 | CHECK | - |
| 8 | 같은 프로젝트에 같은 특허 두 번 | UNIQUE | 2 |
| 9 | 없는 특허를 후보로 | FK | 1 |
| 10 | 우선순위 6 | CHECK (1~5) | - |
| 11 | 최종 결론 없이 상태만 '결론' | CHECK | 8 |
| 12 | 최종 결론에 '보류' | CHECK (채택·제외) | 8 |
| 13 | 결정자만 있고 사유 없음 | CHECK (둘 다 있거나 둘 다 없음) | 8 |
| 14 | 같은 검토자 두 번 배정 | PK | 4 |
| 15 | 배정 안 된 팀원이 검토 작성 | 복합 FK | 4 |
| 16 | 3차 검토 | CHECK (1·2차) | 8 |
| 17 | 결론 없이 제출 | CHECK | 6 |
| 18 | 같은 검토자·같은 차수 검토 두 번 | UNIQUE | 8 |
| 19 | 판정에 영문 대문자 'O' | CHECK (○·△·X) | 6 |
| 20 | 같은 검토에서 같은 비교요소 두 번 판정 | PK | 6 |
| 21 | 근거가 빈 판정 | CHECK | 검토 가이드 4장 |
| 22 | 검토 기록이 있는 후보 삭제 | FK (기록 보호) | 7 |
| 23 | 담당 프로젝트가 있는 팀원 삭제 | FK (기록 보호) | 1 |

In [3]:
tests = [
 ('학번이 8자리 숫자가 아님',        "INSERT INTO team_member(name, student_no) VALUES ('가상', '1234567')"),
 ("프로젝트 상태 '완료'",             "INSERT INTO project(owner_id, title, status) VALUES (3, 't', '완료')"),
 ('없는 팀원을 담당자로',            "INSERT INTO project(owner_id, title) VALUES (99, 't')"),
 ('비교요소 번호 5',                 "INSERT INTO criterion(project_id, no, name) VALUES (1, 5, 't')"),
 ('같은 프로젝트에 같은 번호 비교요소', "INSERT INTO criterion(project_id, no, name) VALUES (1, 1, '중복')"),
 ("대소문자만 다른 같은 검색어",      "INSERT INTO search_term(criterion_id, term) VALUES (1, 'cnn')"),
 ('빈 검색어',                       "INSERT INTO search_term(criterion_id, term) VALUES (1, '   ')"),
 ('같은 프로젝트에 같은 특허 두 번',   f"INSERT INTO candidate(project_id, patent_id) VALUES (1, {P1})"),
 ('없는 특허를 후보로',              "INSERT INTO candidate(project_id, patent_id) VALUES (1, 999999)"),
 ('우선순위 6',                      f"INSERT INTO candidate(project_id, patent_id, priority) VALUES (1, {P3}, 6)"),
 ("최종 결론 없이 상태만 '결론'",     "UPDATE candidate SET status = '결론' WHERE candidate_id = 2"),
 ("최종 결론에 '보류'",               "UPDATE candidate SET final_conclusion = '보류' WHERE candidate_id = 2"),
 ('결정자만 있고 사유 없음',          "UPDATE candidate SET decided_by = 3 WHERE candidate_id = 2"),
 ('같은 검토자 두 번 배정',           "INSERT INTO candidate_reviewer VALUES (1, 1)"),
 ('배정 안 된 팀원이 검토 작성',      "INSERT INTO review(candidate_id, member_id, round) VALUES (1, 3, 1)"),
 ('3차 검토',                        "INSERT INTO review(candidate_id, member_id, round) VALUES (1, 1, 3)"),
 ('결론 없이 제출',                  "UPDATE review SET status = 'SUBMITTED' WHERE review_id = 2"),
 ('같은 검토자·같은 차수 검토 두 번',  "INSERT INTO review(candidate_id, member_id, round) VALUES (1, 1, 1)"),
 ("판정에 영문 대문자 'O'",           "INSERT INTO review_item VALUES (2, 3, 'O', '근거')"),
 ('같은 검토에서 같은 비교요소 두 번', "INSERT INTO review_item VALUES (1, 1, '○', '근거')"),
 ('근거가 빈 판정',                  "INSERT INTO review_item VALUES (2, 4, '○', '')"),
 ('검토 기록이 있는 후보 삭제',       "DELETE FROM candidate WHERE candidate_id = 1"),
 ('담당 프로젝트가 있는 팀원 삭제',    "DELETE FROM team_member WHERE member_id = 3"),
]

ok = 0
for i, (name, sql) in enumerate(tests, 1):
    try:
        con.execute(sql)
        print(f'FAIL(통과됨) {i:2d}. {name}')
    except sqlite3.IntegrityError as e:
        ok += 1
        print(f'PASS(거부)  {i:2d}. {name} - {e}')
    con.rollback()                                  # 시험한 입력 취소
print(f'\n{ok}/{len(tests)} 규칙 정상')

PASS(거부)   1. 학번이 8자리 숫자가 아님 - CHECK constraint failed: student_no IS NULL
                    OR (length(student_no) = 8 AND student_no NOT GLOB '*[^0-9]*')
PASS(거부)   2. 프로젝트 상태 '완료' - CHECK constraint failed: status IN ('준비','진행','확정')
PASS(거부)   3. 없는 팀원을 담당자로 - FOREIGN KEY constraint failed
PASS(거부)   4. 비교요소 번호 5 - CHECK constraint failed: no BETWEEN 1 AND 4
PASS(거부)   5. 같은 프로젝트에 같은 번호 비교요소 - UNIQUE constraint failed: criterion.project_id, criterion.no
PASS(거부)   6. 대소문자만 다른 같은 검색어 - UNIQUE constraint failed: search_term.criterion_id, search_term.term
PASS(거부)   7. 빈 검색어 - CHECK constraint failed: length(trim(term)) > 0
PASS(거부)   8. 같은 프로젝트에 같은 특허 두 번 - UNIQUE constraint failed: candidate.project_id, candidate.patent_id
PASS(거부)   9. 없는 특허를 후보로 - FOREIGN KEY constraint failed
PASS(거부)  10. 우선순위 6 - CHECK constraint failed: priority BETWEEN 1 AND 5
PASS(거부)  11. 최종 결론 없이 상태만 '결론' - CHECK constraint failed: status <> '결론' OR final_conclusion IS NOT NULL
PASS(거부)  12. 최종 결론에 '보류' 

## 4. 지워도 되는 것은 함께 지워지는가 (삭제 정책 확인)
기록이 없는 것은 지울 수 있어야 하고, 지울 때 딸린 행도 같이 지워져야 합니다. 확인 후 모두 되돌립니다.

In [4]:
def cnt(sql):
    return con.execute(sql).fetchone()[0]

checks = []
# (1) 검토 기록 없는 후보 삭제 → 검토자 배정도 함께 삭제
con.execute('DELETE FROM candidate WHERE candidate_id = 2')
checks.append(('검토 기록 없는 후보 삭제 → 배정 2행도 삭제', cnt('SELECT COUNT(*) FROM candidate_reviewer WHERE candidate_id = 2') == 0))
con.rollback()
# (2) 임시(DRAFT) 검토 삭제 → 그 검토의 판정도 함께 삭제
con.execute('DELETE FROM review WHERE review_id = 2')
checks.append(('임시 검토 삭제 → 판정 2행도 삭제', cnt('SELECT COUNT(*) FROM review_item WHERE review_id = 2') == 0))
con.rollback()
# (3) 후보 없는 프로젝트 삭제 → 비교요소·검색어도 함께 삭제
con.execute("INSERT INTO project(project_id, owner_id, title) VALUES (2, 1, '지울 프로젝트')")
con.execute("INSERT INTO criterion(criterion_id, project_id, no, name) VALUES (99, 2, 1, 't')")
con.execute("INSERT INTO search_term(criterion_id, term) VALUES (99, '검색어')")
con.execute('DELETE FROM project WHERE project_id = 2')
checks.append(('후보 없는 프로젝트 삭제 → 비교요소·검색어도 삭제',
               cnt('SELECT COUNT(*) FROM criterion WHERE criterion_id = 99') == 0
               and cnt('SELECT COUNT(*) FROM search_term WHERE criterion_id = 99') == 0))
con.rollback()

for name, passed in checks:
    print('PASS' if passed else 'FAIL', '-', name)
print(f"{sum(p for _, p in checks)}/{len(checks)} 삭제 정책 정상")

PASS - 검토 기록 없는 후보 삭제 → 배정 2행도 삭제
PASS - 임시 검토 삭제 → 판정 2행도 삭제
PASS - 후보 없는 프로젝트 삭제 → 비교요소·검색어도 삭제
3/3 삭제 정책 정상


## 5. DB가 처음 상태 그대로인지 확인

In [5]:
for t in ['patent', 'applicant', 'patent_applicant', 'patent_ipc',
          'team_member', 'project', 'criterion', 'search_term', 'candidate', 'candidate_reviewer', 'review', 'review_item']:
    print(f'{t:20s}', con.execute(f'SELECT COUNT(*) FROM {t}').fetchone()[0])
print('FK 오류:', len(con.execute('PRAGMA foreign_key_check').fetchall()), '/ integrity_check:', con.execute('PRAGMA integrity_check').fetchone()[0])
con.close()

patent               3122
applicant            1246
patent_applicant     3400
patent_ipc           15250
team_member          5
project              1
criterion            4
search_term          2
candidate            2
candidate_reviewer   4
review               2
review_item          6
FK 오류: 0 / integrity_check: ok
